# 09 — Architecture comparison

**In short:** five more ImageNet-pretrained networks are fine-tuned exactly like the
EfficientNet-B0 baseline (notebook 08), so the only thing that changes between runs is
the architecture. Each run is scored the same way and saved, and notebook 10 puts them
side by side and tests an ensemble of the best ones.

| Architecture | Family | Why it is compared |
|---|---|---|
| `efficientnet_b0` | efficient CNN | baseline, trained in notebook 08 |
| `mobilenet_v3_large` | mobile CNN | smallest and fastest; candidate for running on the phone |
| `resnet50` | classic CNN | the standard reference in plant disease papers |
| `densenet121` | dense CNN | widely used in plant disease papers |
| `convnext_tiny` | modern CNN | strongest recent CNN design |
| `swin_t` | vision transformer | a different family from CNNs |

**Fair comparison:** same split, image size, augmentation, epochs, early stopping,
learning rate and seed for every run.

**Winning rule, fixed before any results:** the highest macro-F1 on the *field*
validation photos (FieldPlant + Tanzania) wins, with calibration error (ECE) as the
tie-breaker, provided size and CPU speed stay usable for the backend. PlantDoc is
reported but never used to choose.

**How to run:** set `ARCHITECTURE` in the settings cell, then *Run all*. One architecture
per run. To train the next one in the same session, change `ARCHITECTURE` and run again
from the settings cell; the image copy is reused. Progress is saved after every epoch,
so a disconnect resumes where it stopped.

**Needs a GPU runtime (T4).**

In [ ]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

In [ ]:
# Keep the Drive copy of the repository in sync with GitHub.
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

In [ ]:
%pip install -q -r {ML_DIR}/requirements.txt

## Settings

Pick one of `mobilenet_v3_large`, `resnet50`, `densenet121`, `convnext_tiny`, `swin_t`.
Everything else is copied from the baseline and must not change between runs.

`LEARNING_RATE` stays at the shared 1e-3. As planned in #41, a model that clearly fails to
train at 1e-3 is trained **again** at a lower rate; that rerun is saved as a separate run
(e.g. `compare_convnext_tiny_lr1e-04`) so the result at the shared settings is kept and both
are reported. Set `FRESH_START = True` only to throw away an earlier attempt of the same run.

In [ ]:
import importlib
import shutil

import torch

from src import training
importlib.reload(training)

ARCHITECTURE = "mobilenet_v3_large"
LEARNING_RATE = 1e-3
FRESH_START = False

assert ARCHITECTURE in training.ARCHITECTURES, f"Unknown architecture: {ARCHITECTURE}"
assert ARCHITECTURE != "efficientnet_b0", "EfficientNet-B0 is the baseline from notebook 08; no need to train it again."

RUN_NAME = f"compare_{ARCHITECTURE}" + ("" if LEARNING_RATE == 1e-3 else f"_lr{LEARNING_RATE:.0e}")
TRAIN_SOURCES = ["plantvillage", "fieldplant", "tanzania"]
FIELD_SOURCES = ["fieldplant", "tanzania"]

# Identical to notebook 08 apart from the architecture (and the learning rate for a planned rerun).
SETTINGS = {
    "architecture": f"{ARCHITECTURE} (ImageNet weights: {training.ARCHITECTURES[ARCHITECTURE][1].name})",
    "image_size": 224,
    "batch_size": 64,
    "epochs": 15,
    "patience": 4,
    "learning_rate": LEARNING_RATE,
    "weight_decay": 1e-2,
    "seed": config.RANDOM_SEED,
    "train_sources": TRAIN_SOURCES,
}

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again."
DEVICE = "cuda"
SETTINGS["gpu"] = torch.cuda.get_device_name(0)
SETTINGS["torch"] = torch.__version__

RUN_DIR = config.MODELS_DIR / RUN_NAME
if FRESH_START and RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)
training.seed_everything(config.RANDOM_SEED)
print("Architecture:", ARCHITECTURE, "| learning rate:", LEARNING_RATE, "| run folder:", RUN_DIR, "| GPU:", SETTINGS["gpu"])

## Load the split

The same CSVs as the baseline. Test keeps every source so results can be compared per source.

In [ ]:
import pandas as pd

splits = {name: pd.read_csv(config.SPLITS_DIR / f"{name}.csv") for name in ["train", "val", "test", "ood_test", "unseen_test"]}
for name in ["train", "val"]:
    splits[name] = splits[name][splits[name]["source"].isin(TRAIN_SOURCES)]

CLASSES = list(config.PLANTVILLAGE_CLASSES)
assert set(splits["train"]["class"]) == set(CLASSES), "Every class must have training images"
display(pd.DataFrame({name: rows["source"].value_counts() for name, rows in splits.items()}).fillna(0).astype(int))

## Copy the images to local disk

About 10 minutes the first time in a session; instant for the second architecture.

In [ ]:
from pathlib import Path

from src.data import image_dataset

LOCAL_ROOT = Path("/content/image_cache")
all_paths = sorted(set(pd.concat(splits.values())["path"]))
cached = image_dataset.cache_images(all_paths, config.RAW_DATA_DIR, LOCAL_ROOT)
assert cached == len(all_paths), f"Only {cached} of {len(all_paths)} images cached"
print("Cached", cached, "images in", LOCAL_ROOT)

In [ ]:
from torch.utils.data import DataLoader

size = SETTINGS["image_size"]
datasets = {
    name: image_dataset.SplitDataset(
        rows, LOCAL_ROOT, CLASSES,
        image_dataset.train_transform(size) if name == "train" else image_dataset.eval_transform(size),
    )
    for name, rows in splits.items()
}
loaders = {
    name: DataLoader(ds, batch_size=SETTINGS["batch_size"], shuffle=name == "train", drop_last=name == "train",
                     num_workers=2, pin_memory=True, persistent_workers=True)
    for name, ds in datasets.items()
}
print({name: len(ds) for name, ds in datasets.items()})

## Train

Each line is one epoch. Small models take about 1–3 minutes per epoch on a T4; ConvNeXt and Swin take longer.

In [ ]:
training.save_json(SETTINGS, RUN_DIR / "settings.json")
model = training.build_model(len(CLASSES), ARCHITECTURE)
history = training.fit(model, loaders["train"], loaders["val"], RUN_DIR, SETTINGS, DEVICE)
display(history.round(4))

## Training curves

Train and validation loss should fall together. Validation loss rising while train loss
keeps falling would mean overfitting; early stopping keeps the best epoch either way.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(history["epoch"], history["train_loss"], label="train loss")
ax1.plot(history["epoch"], history["val_loss"], label="val loss")
ax1.set_xlabel("epoch"); ax1.legend(); ax1.set_title(f"{ARCHITECTURE}: loss")
ax2.plot(history["epoch"], history["val_accuracy"], label="val accuracy")
ax2.plot(history["epoch"], history["val_macro_f1"], label="val macro-F1")
ax2.set_xlabel("epoch"); ax2.legend(); ax2.set_title(f"{ARCHITECTURE}: validation")
plt.tight_layout()
plt.savefig(RUN_DIR / "training_curves.png", dpi=150)
plt.show()

## Evaluate the best epoch

The row **`val: field`** is the one the winning rule uses. Test (overall and per source)
and PlantDoc (`ood_test`) are reported for every model the same way. Predictions and raw
scores (logits) are saved for the ensemble and for calibration.

In [ ]:
import numpy as np

from src import metrics

model.load_state_dict(torch.load(RUN_DIR / "best.pt", map_location=DEVICE))
model.to(DEVICE)
RESULTS_DIR = RUN_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

outputs, rows = {}, []
for name in ["val", "test", "ood_test", "unseen_test"]:
    logits, labels = training.predict(model, loaders[name], DEVICE)
    outputs[name] = training.save_predictions(splits[name], logits, CLASSES, RESULTS_DIR / f"{name}_predictions.csv")
    if name == "unseen_test":
        continue
    rows.append({"set": name, **metrics.summarise(logits, labels, len(CLASSES))})
    sources = splits[name]["source"].to_numpy()
    if name == "val":
        field = np.isin(sources, FIELD_SOURCES)
        rows.append({"set": "val: field", **metrics.summarise(logits[field], labels[field], len(CLASSES))})
    if name in ["val", "test"]:
        for source in sorted(set(sources)):
            mask = sources == source
            rows.append({"set": f"{name}: {source}", **metrics.summarise(logits[mask], labels[mask], len(CLASSES))})

results = pd.DataFrame(rows).set_index("set")
display(results.round(4))
results.to_csv(RESULTS_DIR / "metrics.csv")

unseen = outputs["unseen_test"]["confidence"]
print(f"Unseen tomato diseases ({len(unseen)} images): mean confidence {unseen.mean():.3f}, "
      f"{(unseen >= 0.9).mean():.1%} predicted with confidence >= 0.90")

## Per-class results and confusion matrix (test)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

short = [c.replace("Corn_(maize)___", "Maize ").replace("___", " ").replace("_", " ") for c in CLASSES]
test_out = outputs["test"]
report = classification_report(test_out["class"], test_out["predicted"], labels=CLASSES, target_names=short,
                               digits=3, zero_division=0, output_dict=True)
report_table = pd.DataFrame(report).T.round(3)
display(report_table)
report_table.to_csv(RESULTS_DIR / "test_per_class.csv")

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay.from_predictions(test_out["class"], test_out["predicted"], labels=CLASSES,
                                        display_labels=short, xticks_rotation=90, ax=ax, colorbar=False)
ax.set_title(f"{ARCHITECTURE}: test confusion matrix")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "test_confusion_matrix.png", dpi=150)
plt.show()

## Where the model looks (Grad-CAM)

Red areas pushed the model towards its answer. On a good model they sit on the lesions,
not the background. The same photos (fixed seed) are shown for every architecture:
top row field test photos, bottom row PlantDoc.

In [ ]:
from src import explain

def pick(name, sources, count):
    rows = splits[name].reset_index(drop=True)
    if sources:
        rows = rows[rows["source"].isin(sources)]
    rows = rows[rows["class"].isin(CLASSES)]
    return rows.sample(count, random_state=config.RANDOM_SEED).index.tolist()

examples = [("test", i) for i in pick("test", FIELD_SOURCES, 4)] + [("ood_test", i) for i in pick("ood_test", None, 4)]
fig, axes = plt.subplots(2, 4, figsize=(14, 7.5))
for ax, (name, i) in zip(axes.flat, examples):
    image, label = datasets[name][i]
    heatmap, predicted = explain.grad_cam(model, ARCHITECTURE, image.to(DEVICE))
    confidence = outputs[name].loc[i, "confidence"]
    ax.imshow(explain.to_picture(image, image_dataset.MEAN, image_dataset.STD))
    ax.imshow(heatmap, cmap="jet", alpha=0.4)
    mark = "right" if predicted == label else "WRONG"
    ax.set_title(f"true: {short[label]}\npredicted: {short[predicted]} ({confidence:.0%}, {mark})", fontsize=8)
    ax.axis("off")
plt.suptitle(f"{ARCHITECTURE}: Grad-CAM (top: field test photos, bottom: PlantDoc)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "gradcam.png", dpi=120)
plt.show()

## Size and speed

Parameters and file size decide whether the model could later run on a phone; CPU time
per image is what the backend (no GPU) will need. Timed on Colab's CPU with 2 threads,
so compare models with each other rather than with the laptop.

In [ ]:
summary = {
    "architecture": ARCHITECTURE,
    "learning_rate": LEARNING_RATE,
    "parameters_millions": training.count_parameters(model) / 1e6,
    "best_epoch": int(history.loc[history["val_macro_f1"].idxmax(), "epoch"]),
    "epochs_run": len(history),
    "minutes_training": float(history["minutes"].sum()),
}
training.export(model.cpu(), CLASSES, SETTINGS["image_size"], image_dataset.MEAN, image_dataset.STD,
                RUN_DIR / "model.pt", SETTINGS, architecture=ARCHITECTURE)
summary["file_mb"] = (RUN_DIR / "model.pt").stat().st_size / 1e6
summary["cpu_ms_per_image"] = training.cpu_milliseconds_per_image(model, SETTINGS["image_size"])
training.save_json(summary, RESULTS_DIR / "summary.json")
display(pd.Series(summary))

## Copy the results into the repository

Weights stay on Drive; the small result files (metrics, curves, confusion matrix,
Grad-CAM, summary) go to `ml/results/<run>/` so they are visible on GitHub.

In [ ]:
REPO_RESULTS = ML_DIR / "results" / RUN_NAME
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
for file in [RUN_DIR / "settings.json", RUN_DIR / "history.csv", RUN_DIR / "training_curves.png",
             RESULTS_DIR / "metrics.csv", RESULTS_DIR / "test_per_class.csv", RESULTS_DIR / "test_confusion_matrix.png",
             RESULTS_DIR / "gradcam.png", RESULTS_DIR / "summary.json"]:
    shutil.copy(file, REPO_RESULTS / file.name)
print(sorted(p.name for p in REPO_RESULTS.iterdir()))

## Make sure Drive has saved everything

Colab uploads files to Drive in the background; this waits until the upload has finished
(see DEF-01), then reconnects Drive for the push.

In [ ]:
drive.flush_and_unmount()
drive.mount("/content/drive")
print("Drive synced and remounted.")

## Save to GitHub

Save the notebook first (Ctrl+S). This keeps a copy of the run's notebook with its
outputs in the results folder and commits **only that folder** to a new branch, so this
notebook stays a clean template and the pull requests for different architectures never
conflict. Merge each pull request on GitHub as usual.

In [ ]:
from google.colab import userdata

from src import github_sync

shutil.copy(ML_DIR / "notebooks" / "09_architecture_comparison.ipynb", REPO_RESULTS / "notebook.ipynb")
print(github_sync.status(REPO_DIR))
print()
print(github_sync.push(
    REPO_DIR,
    branch=f"colab/{RUN_NAME.replace('_', '-')}",
    message=f"Train and evaluate {RUN_NAME.removeprefix('compare_')} for the architecture comparison",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))